# Estrutura da tabela

In [0]:
DESCRIBE dev_procurement.corp_curated.vw_ds_log_mb25;

# volumetria, granularidade e unicidade

In [0]:
-- Baseline de volumetria e teste inicial de granularidade
SELECT
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT num_reserva) AS reservas_distintas,
    COUNT(DISTINCT cod_material) AS materiais_distintos,
    COUNT(DISTINCT cod_centro) AS centros_distintos,
    COUNT(DISTINCT cod_deposito) AS depositos_distintos,
    COUNT(
        DISTINCT CONCAT_WS(
            '|',
            COALESCE(num_reserva, '<NULL>'),
            COALESCE(num_item_reserva, '<NULL>')
        )
    ) AS reserva_item_distintos
FROM dev_procurement.corp_curated.vw_ds_log_mb25;

# Teste Duplicidade

In [0]:
-- Teste de unicidade da chave candidata reserva + item
SELECT
    num_reserva,
    num_item_reserva,
    COUNT(*) AS qtd_linhas,
    COUNT(DISTINCT cod_material) AS materiais,
    COUNT(DISTINCT cod_centro) AS centros,
    COUNT(DISTINCT cod_deposito) AS depositos,
    COUNT(DISTINCT num_lote) AS lotes
FROM dev_procurement.corp_curated.vw_ds_log_mb25
GROUP BY
    num_reserva,
    num_item_reserva
HAVING COUNT(*) > 1
ORDER BY
    qtd_linhas DESC,
    num_reserva,
    num_item_reserva
LIMIT 100;

#completude de TODAS as colunas

In [0]:
SELECT *
FROM dev_procurement.corp_curated.vw_ds_log_mb25
LIMIT 20;

In [0]:
SELECT
    MIN(dt_reserva) AS min_dt_reserva,
    MAX(dt_reserva) AS max_dt_reserva,
    MIN(dt_necessidade) AS min_dt_necessidade,
    MAX(dt_necessidade) AS max_dt_necessidade,
    MIN(dt_criacao) AS min_dt_criacao,
    MAX(dt_criacao) AS max_dt_criacao
FROM dev_procurement.corp_curated.vw_ds_log_mb25;

# Recomendação antes da Fase 2
A metodologia exige verificar preenchimento das colunas antes de propor cenários SAP

In [0]:
SELECT
    COUNT(*) total,
    SUM(CASE WHEN desc_material IS NULL OR TRIM(desc_material) IN ('', 'null') THEN 1 ELSE 0 END) qtd_desc_material_vazia,
    SUM(CASE WHEN dt_reserva = '00000000' THEN 1 ELSE 0 END) qtd_dt_reserva_zerada,
    SUM(CASE WHEN cod_deposito IS NULL OR TRIM(cod_deposito) = '' THEN 1 ELSE 0 END) qtd_deposito_vazio,
    SUM(CASE WHEN num_lote IS NULL OR TRIM(num_lote) = '' THEN 1 ELSE 0 END) qtd_lote_vazio,
    SUM(CASE WHEN nm_usuario IS NULL OR TRIM(nm_usuario) = '' THEN 1 ELSE 0 END) qtd_usuario_vazio
FROM dev_procurement.corp_curated.vw_ds_log_mb25;

# Analise centro volmetria

In [0]:
SELECT
    cod_centro,
    COUNT(*) AS total_linhas,
    COUNT(DISTINCT num_reserva) AS reservas,
    COUNT(DISTINCT cod_material) AS materiais,

    SUM(
        CASE
            WHEN desc_material IS NULL
              OR TRIM(desc_material) IN ('', 'null')
            THEN 1 ELSE 0
        END
    ) AS desc_material_vazia,

    SUM(
        CASE
            WHEN dt_reserva = '00000000'
              OR dt_reserva IS NULL
              OR TRIM(dt_reserva) = ''
            THEN 1 ELSE 0
        END
    ) AS dt_reserva_invalida,

    SUM(
        CASE
            WHEN cod_deposito IS NULL
              OR TRIM(cod_deposito) = ''
            THEN 1 ELSE 0
        END
    ) AS deposito_vazio,

    SUM(
        CASE
            WHEN nm_usuario IS NULL
              OR TRIM(nm_usuario) = ''
            THEN 1 ELSE 0
        END
    ) AS usuario_vazio,

    SUM(
        CASE
            WHEN ind_item_eliminado = 'X'
            THEN 1 ELSE 0
        END
    ) AS itens_eliminados,

    SUM(
        CASE
            WHEN ind_registro_final = 'X'
            THEN 1 ELSE 0
        END
    ) AS registros_finais,

    MIN(
        CASE
            WHEN dt_necessidade RLIKE '^[0-9]{8}$'
             AND dt_necessidade <> '00000000'
            THEN dt_necessidade
        END
    ) AS min_dt_necessidade,

    MAX(
        CASE
            WHEN dt_necessidade RLIKE '^[0-9]{8}$'
             AND dt_necessidade <> '00000000'
            THEN dt_necessidade
        END
    ) AS max_dt_necessidade

FROM dev_procurement.corp_curated.vw_ds_log_mb25
GROUP BY cod_centro
ORDER BY total_linhas DESC;